## 准备数据

In [11]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [12]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [ ]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################

        self.W1 = tf.Variable(tf.random.normal([784, 100], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([100]))
        self.W2 = tf.Variable(tf.random.normal([100, 10], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([10]))

    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################

        # 将28x28的图片展平为784维向量
        x = tf.reshape(x, [-1, 784])
        
        # 第一层: 线性变换 + ReLU激活
        h1 = tf.matmul(x, self.W1) + self.b1
        h1_relu = tf.nn.relu(h1)
        
        # 第二层: 线性变换，输出logits
        logits = tf.matmul(h1_relu, self.W2) + self.b2
        
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [19]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [20]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.358287 ; accuracy 0.1185
epoch 1 : loss 2.3539162 ; accuracy 0.12066667
epoch 2 : loss 2.349615 ; accuracy 0.12245
epoch 3 : loss 2.3453832 ; accuracy 0.12448333
epoch 4 : loss 2.3412187 ; accuracy 0.12701666
epoch 5 : loss 2.3371181 ; accuracy 0.12903333
epoch 6 : loss 2.3330796 ; accuracy 0.13133334
epoch 7 : loss 2.3290987 ; accuracy 0.13341667
epoch 8 : loss 2.3251748 ; accuracy 0.13568333
epoch 9 : loss 2.3213022 ; accuracy 0.13808334
epoch 10 : loss 2.3174784 ; accuracy 0.1401
epoch 11 : loss 2.313699 ; accuracy 0.14241667
epoch 12 : loss 2.3099644 ; accuracy 0.14478333
epoch 13 : loss 2.3062754 ; accuracy 0.14716667
epoch 14 : loss 2.302628 ; accuracy 0.14951667
epoch 15 : loss 2.2990234 ; accuracy 0.15181667
epoch 16 : loss 2.2954617 ; accuracy 0.1546
epoch 17 : loss 2.2919402 ; accuracy 0.1571
epoch 18 : loss 2.2884574 ; accuracy 0.15895
epoch 19 : loss 2.285012 ; accuracy 0.1614
epoch 20 : loss 2.281603 ; accuracy 0.16403334
epoch 21 : loss 2.2782254 ; accura